# 🌍 Cross-Linguistic Profanity & Slang Dataset — Full Sociolinguistic Analysis

A comprehensive exploratory data analysis of ~100K profanity, slang, and taboo language entries across dozens of languages, scripts, and cultures.

**Sections:**
1. Setup & Data Loading
2. Dataset Overview & Quality Audit
3. Linguistic Landscape (languages, scripts, regions)
4. Category Taxonomy Analysis
5. Severity & Toxicity Analysis
6. Popularity & Generational Analysis
7. Content Flag Analysis (hate speech, sexual, religious, family)
8. Etymology & Loanword Patterns
9. Word-Level Text Analysis (length, script, phonetics)
10. Cross-Linguistic Semantic Clustering (Embeddings)
11. Network / Co-occurrence Analysis
12. Statistical Tests & Correlations
13. Paper-Ready Summary Tables & Figures

---
## 1. Setup & Data Loading

In [1]:
# ============================================================
# 1A — Install dependencies (run once)
# ============================================================
# !pip install pandas numpy matplotlib seaborn scipy scikit-learn
# !pip install wordcloud plotly kaleido
# !pip install sentence-transformers   # optional: for embedding clustering

import warnings
warnings.filterwarnings('ignore')

import json
import re
import os
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder

# Style
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 200,
    'figure.figsize': (12, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

# Output directory for plots
PLOT_DIR = Path('plots')
PLOT_DIR.mkdir(exist_ok=True)

print('✅ All imports successful.')

✅ All imports successful.


In [2]:
# ============================================================
# 1B — Load Data
# ============================================================
# Supports: JSONL, JSON array, or CSV
#
# >>> EDIT THIS PATH to point to your actual file <<<
DATA_PATH = 'profanity_dataset.jsonl'   # or .json / .csv

def load_data(path: str) -> pd.DataFrame:
    """Auto-detect format and load."""
    p = Path(path)
    if p.suffix == '.csv':
        return pd.read_csv(p)
    
    # Try JSONL first (one JSON object per line)
    try:
        records = []
        with open(p, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        if records:
            return pd.DataFrame(records)
    except json.JSONDecodeError:
        pass
    
    # Try JSON array
    with open(p, 'r', encoding='utf-8') as f:
        data = json.load(f)
    if isinstance(data, list):
        return pd.DataFrame(data)
    raise ValueError(f'Cannot parse {path}')

df = load_data(DATA_PATH)
print(f'Loaded {len(df):,} entries with {df.shape[1]} columns.')
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'profanity_dataset.jsonl'

---
## 2. Dataset Overview & Quality Audit

In [ ]:
# ============================================================
# 2A — Shape, dtypes, memory
# ============================================================
print(f'Rows:    {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')
print(f'Memory:  {df.memory_usage(deep=True).sum() / 1e6:.1f} MB\n')
df.info(memory_usage='deep')

In [ ]:
# ============================================================
# 2B — Missing values heatmap
# ============================================================
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df.missing_count > 0].sort_values('missing_pct', ascending=False)

if len(missing_df):
    fig, ax = plt.subplots(figsize=(10, max(3, len(missing_df)*0.4)))
    ax.barh(missing_df.index, missing_df['missing_pct'], color='#e74c3c')
    ax.set_xlabel('% Missing')
    ax.set_title('Missing Values by Column')
    for i, (_, row) in enumerate(missing_df.iterrows()):
        ax.text(row['missing_pct']+0.3, i, f"{row['missing_count']:,} ({row['missing_pct']}%)", va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'missing_values.png')
    plt.show()
else:
    print('✅ No missing values!')

print('\n--- Missing summary ---')
display(missing_df) if len(missing_df) else None

In [ ]:
# ============================================================
# 2C — Duplicate check
# ============================================================
dup_id = df['id'].duplicated().sum() if 'id' in df.columns else 'N/A'
dup_word = df.duplicated(subset=['language', 'word']).sum()
dup_full = df.duplicated().sum()

print(f'Duplicate IDs:                  {dup_id}')
print(f'Duplicate (language + word):     {dup_word}')
print(f'Fully duplicate rows:            {dup_full}')

In [ ]:
# ============================================================
# 2D — Ensure boolean columns are bool
# ============================================================
bool_cols = ['safe_for_work', 'hate_speech', 'sexual', 'religious', 'family_related']
for c in bool_cols:
    if c in df.columns:
        df[c] = df[c].astype(bool)

# Ensure numerics
for c in ['severity', 'popularity_score']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

print('✅ dtypes cleaned.')
df.dtypes

In [ ]:
# ============================================================
# 2E — Quick descriptive stats for numeric columns
# ============================================================
df.describe(include='all').T

---
## 3. Linguistic Landscape

In [ ]:
# ============================================================
# 3A — Key counts
# ============================================================
summary = {
    'Total entries': len(df),
    'Unique languages': df['language'].nunique(),
    'Unique countries': df['country'].nunique(),
    'Unique regions': df['region'].nunique(),
    'Unique scripts': df['script'].nunique(),
    'Unique categories': df['category'].nunique(),
    'Unique words': df['word'].nunique(),
    'Unique target_types': df['target_type'].nunique(),
    'Unique tones': df['tone'].nunique(),
    'Unique generations': df['generation'].nunique(),
}
summary_df = pd.DataFrame.from_dict(summary, orient='index', columns=['Count'])
display(summary_df)

In [ ]:
# ============================================================
# 3B — Entries per language (top 30)
# ============================================================
lang_counts = df['language'].value_counts()

fig, ax = plt.subplots(figsize=(14, 8))
top_n = min(30, len(lang_counts))
lang_counts.head(top_n).plot.barh(ax=ax, color=sns.color_palette('viridis', top_n))
ax.invert_yaxis()
ax.set_xlabel('Number of Entries')
ax.set_title(f'Top {top_n} Languages by Entry Count')
for i, v in enumerate(lang_counts.head(top_n)):
    ax.text(v + lang_counts.max()*0.005, i, f'{v:,}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'entries_per_language.png')
plt.show()

print(f'\nLanguages with only 1 entry: {(lang_counts == 1).sum()}')
print(f'Languages with 100+ entries: {(lang_counts >= 100).sum()}')

In [ ]:
# ============================================================
# 3C — Entries per script
# ============================================================
script_counts = df['script'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
script_counts.head(15).plot.barh(ax=axes[0], color=sns.color_palette('magma', 15))
axes[0].invert_yaxis()
axes[0].set_title('Entries per Script (Top 15)')
axes[0].set_xlabel('Count')

# Pie chart for script families
top5 = script_counts.head(5)
other = pd.Series({'Other': script_counts.iloc[5:].sum()}) if len(script_counts) > 5 else pd.Series()
pie_data = pd.concat([top5, other])
axes[1].pie(pie_data, labels=pie_data.index, autopct='%1.1f%%', startangle=140,
            colors=sns.color_palette('Set2', len(pie_data)))
axes[1].set_title('Script Distribution')

plt.tight_layout()
plt.savefig(PLOT_DIR / 'script_distribution.png')
plt.show()

In [ ]:
# ============================================================
# 3D — Entries per country (top 20)
# ============================================================
country_counts = df['country'].value_counts()

fig, ax = plt.subplots(figsize=(14, 7))
top_n = min(20, len(country_counts))
country_counts.head(top_n).plot.barh(ax=ax, color=sns.color_palette('rocket', top_n))
ax.invert_yaxis()
ax.set_xlabel('Number of Entries')
ax.set_title(f'Top {top_n} Countries by Entry Count')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'entries_per_country.png')
plt.show()

In [ ]:
# ============================================================
# 3E — Language × Script matrix (heatmap)
# ============================================================
lang_script = pd.crosstab(df['language'], df['script'])

# Filter to top languages and scripts for readability
top_langs = df['language'].value_counts().head(20).index
top_scripts = df['script'].value_counts().head(10).index
subset = lang_script.loc[
    lang_script.index.isin(top_langs),
    lang_script.columns.isin(top_scripts)
]

if subset.shape[0] > 1 and subset.shape[1] > 1:
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(subset, annot=True, fmt='d', cmap='YlOrRd', ax=ax, linewidths=0.5)
    ax.set_title('Language × Script Cross-Tab (top languages & scripts)')
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'language_script_heatmap.png')
    plt.show()

In [ ]:
# ============================================================
# 3F — Languages per country (diversity index)
# ============================================================
langs_per_country = df.groupby('country')['language'].nunique().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
langs_per_country.head(20).plot.bar(ax=ax, color=sns.color_palette('crest', 20))
ax.set_ylabel('Number of distinct languages')
ax.set_title('Linguistic Diversity: Languages per Country (top 20)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'languages_per_country.png')
plt.show()

---
## 4. Category Taxonomy Analysis

In [ ]:
# ============================================================
# 4A — Category distribution
# ============================================================
cat_counts = df['category'].value_counts()
print(f'Total unique categories: {len(cat_counts)}\n')

fig, ax = plt.subplots(figsize=(14, max(6, len(cat_counts.head(30))*0.35)))
top_n = min(30, len(cat_counts))
colors = sns.color_palette('husl', top_n)
cat_counts.head(top_n).plot.barh(ax=ax, color=colors)
ax.invert_yaxis()
ax.set_xlabel('Count')
ax.set_title(f'Top {top_n} Profanity Categories')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'category_distribution.png')
plt.show()

In [ ]:
# ============================================================
# 4B — Category × Language heatmap (top 15 each)
# ============================================================
top_cats = df['category'].value_counts().head(15).index
top_langs = df['language'].value_counts().head(15).index

ct = pd.crosstab(df['language'], df['category'])
subset = ct.loc[
    ct.index.isin(top_langs),
    ct.columns.isin(top_cats)
]

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(subset, annot=True, fmt='d', cmap='Blues', ax=ax, linewidths=0.5)
ax.set_title('Category × Language (top 15 × 15)')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'category_language_heatmap.png')
plt.show()

In [ ]:
# ============================================================
# 4C — Target type distribution
# ============================================================
target_counts = df['target_type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

target_counts.plot.bar(ax=axes[0], color=sns.color_palette('Set2', len(target_counts)))
axes[0].set_title('Target Type Distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

axes[1].pie(target_counts, labels=target_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set2', len(target_counts)), startangle=90)
axes[1].set_title('Target Type Proportions')

plt.tight_layout()
plt.savefig(PLOT_DIR / 'target_type_dist.png')
plt.show()

In [ ]:
# ============================================================
# 4D — Tone distribution
# ============================================================
tone_counts = df['tone'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
tone_counts.plot.bar(ax=ax, color=sns.color_palette('coolwarm', len(tone_counts)))
ax.set_title('Tone Distribution')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'tone_distribution.png')
plt.show()

In [ ]:
# ============================================================
# 4E — Category × Tone heatmap
# ============================================================
ct_tone = pd.crosstab(df['category'], df['tone'])
top_cat_tone = ct_tone.loc[ct_tone.sum(axis=1).nlargest(15).index]

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(top_cat_tone, annot=True, fmt='d', cmap='Oranges', ax=ax, linewidths=0.5)
ax.set_title('Category × Tone (top 15 categories)')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'category_tone_heatmap.png')
plt.show()

---
## 5. Severity & Toxicity Analysis

In [ ]:
# ============================================================
# 5A — Severity distribution
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
df['severity'].hist(bins=range(1, df['severity'].max()+2), ax=axes[0],
                     color='#e74c3c', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Severity')
axes[0].set_ylabel('Count')
axes[0].set_title('Severity Distribution')
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Value counts
sev_vc = df['severity'].value_counts().sort_index()
sev_vc.plot.bar(ax=axes[1], color=sns.color_palette('Reds_d', len(sev_vc)))
axes[1].set_xlabel('Severity Level')
axes[1].set_ylabel('Count')
axes[1].set_title('Entries per Severity Level')
for i, v in enumerate(sev_vc):
    axes[1].text(i, v + sev_vc.max()*0.01, f'{v:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'severity_distribution.png')
plt.show()

print(f"Mean severity: {df['severity'].mean():.2f}")
print(f"Median severity: {df['severity'].median():.1f}")
print(f"Std severity: {df['severity'].std():.2f}")

In [ ]:
# ============================================================
# 5B — Average severity by category (top 20)
# ============================================================
sev_by_cat = df.groupby('category')['severity'].agg(['mean','median','std','count'])
sev_by_cat = sev_by_cat[sev_by_cat['count'] >= 5]  # filter low-n categories
sev_by_cat_sorted = sev_by_cat.sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
top_n = min(25, len(sev_by_cat_sorted))
bars = ax.barh(range(top_n), sev_by_cat_sorted['mean'].head(top_n),
               xerr=sev_by_cat_sorted['std'].head(top_n), capsize=3,
               color=sns.color_palette('flare', top_n), edgecolor='white')
ax.set_yticks(range(top_n))
ax.set_yticklabels(sev_by_cat_sorted.index[:top_n])
ax.invert_yaxis()
ax.set_xlabel('Mean Severity (± std)')
ax.set_title('Mean Severity by Category (min 5 entries, top 25)')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'severity_by_category.png')
plt.show()

print('\nFull table (top 15):')
display(sev_by_cat_sorted.head(15).round(2))

In [ ]:
# ============================================================
# 5C — Average severity by language (top 20)
# ============================================================
sev_by_lang = df.groupby('language')['severity'].agg(['mean','median','count'])
sev_by_lang = sev_by_lang[sev_by_lang['count'] >= 5]
sev_by_lang_sorted = sev_by_lang.sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
top_n = min(20, len(sev_by_lang_sorted))
sev_by_lang_sorted['mean'].head(top_n).plot.barh(ax=ax, color=sns.color_palette('rocket', top_n))
ax.invert_yaxis()
ax.set_xlabel('Mean Severity')
ax.set_title('Mean Severity by Language (min 5 entries)')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'severity_by_language.png')
plt.show()

In [ ]:
# ============================================================
# 5D — Boxplot: severity by category (top 12)
# ============================================================
top_cats = df['category'].value_counts().head(12).index
subset = df[df['category'].isin(top_cats)]

fig, ax = plt.subplots(figsize=(14, 7))
order = subset.groupby('category')['severity'].median().sort_values(ascending=False).index
sns.boxplot(data=subset, y='category', x='severity', order=order, ax=ax,
            palette='flare', showfliers=True)
ax.set_title('Severity Distribution by Category (top 12 categories)')
ax.set_xlabel('Severity')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'severity_boxplot_by_category.png')
plt.show()

In [ ]:
# ============================================================
# 5E — Boxplot: severity by tone
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))
order = df.groupby('tone')['severity'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='tone', y='severity', order=order, ax=ax, palette='coolwarm')
ax.set_title('Severity by Tone')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'severity_by_tone.png')
plt.show()

In [ ]:
# ============================================================
# 5F — Scatterplot: popularity_score vs severity
# ============================================================
fig, ax = plt.subplots(figsize=(10, 7))
# Jitter for discrete values
jitter_sev = df['severity'] + np.random.normal(0, 0.1, len(df))
jitter_pop = df['popularity_score'] + np.random.normal(0, 0.1, len(df))

ax.scatter(jitter_sev, jitter_pop, alpha=0.15, s=10, c='#3498db')
ax.set_xlabel('Severity')
ax.set_ylabel('Popularity Score')
ax.set_title('Popularity vs Severity (jittered)')

# Trend line
z = np.polyfit(df['severity'].dropna(), df['popularity_score'].dropna(), 1)
p = np.poly1d(z)
x_line = np.linspace(df['severity'].min(), df['severity'].max(), 100)
ax.plot(x_line, p(x_line), 'r-', linewidth=2, label=f'Trend (slope={z[0]:.2f})')
ax.legend()

# Correlation
r, pval = stats.pearsonr(df['severity'].dropna(), df['popularity_score'].dropna())
ax.text(0.02, 0.98, f'Pearson r = {r:.3f} (p = {pval:.2e})',
        transform=ax.transAxes, va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(PLOT_DIR / 'popularity_vs_severity.png')
plt.show()

In [ ]:
# ============================================================
# 5G — Toxicity composite score
# ============================================================
df['toxicity_score'] = (
    df['severity']
    + 2 * df['hate_speech'].astype(int)
    + df['sexual'].astype(int)
    + df['religious'].astype(int)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
df['toxicity_score'].hist(bins=30, ax=axes[0], color='#9b59b6', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Toxicity Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Toxicity Score Distribution')
axes[0].axvline(df['toxicity_score'].mean(), color='red', linestyle='--', label=f"Mean={df['toxicity_score'].mean():.1f}")
axes[0].legend()

# By category
tox_by_cat = df.groupby('category')['toxicity_score'].mean().sort_values(ascending=False)
tox_by_cat.head(15).plot.barh(ax=axes[1], color=sns.color_palette('Purples_d', 15))
axes[1].invert_yaxis()
axes[1].set_xlabel('Mean Toxicity Score')
axes[1].set_title('Mean Toxicity by Category (top 15)')

plt.tight_layout()
plt.savefig(PLOT_DIR / 'toxicity_score.png')
plt.show()

# Top 20 most toxic entries
print('\n🔴 Top 20 most toxic entries:')
display(df.nlargest(20, 'toxicity_score')[['word','transliteration','language','category','toxicity_score','severity','hate_speech','sexual']])

---
## 6. Popularity & Generational Slang Analysis

In [ ]:
# ============================================================
# 6A — Popularity score distribution
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['popularity_score'].hist(bins=range(1, 12), ax=axes[0], color='#2ecc71', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Popularity Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Popularity Score Distribution')
axes[0].axvline(df['popularity_score'].mean(), color='red', linestyle='--',
                label=f"Mean={df['popularity_score'].mean():.1f}")
axes[0].legend()

# KDE by generation
for gen in df['generation'].unique():
    sub = df[df['generation'] == gen]['popularity_score'].dropna()
    if len(sub) > 10:
        sub.plot.kde(ax=axes[1], label=gen, linewidth=2)
axes[1].set_xlabel('Popularity Score')
axes[1].set_title('Popularity Distribution by Generation (KDE)')
axes[1].set_xlim(0, 11)
axes[1].legend()

plt.tight_layout()
plt.savefig(PLOT_DIR / 'popularity_distribution.png')
plt.show()

In [ ]:
# ============================================================
# 6B — Generation distribution
# ============================================================
gen_counts = df['generation'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

gen_counts.plot.bar(ax=axes[0], color=sns.color_palette('tab10', len(gen_counts)))
axes[0].set_title('Entries per Generation')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(gen_counts):
    axes[0].text(i, v + gen_counts.max()*0.01, f'{v:,}', ha='center', fontsize=9)

# Mean popularity by generation
gen_pop = df.groupby('generation')['popularity_score'].mean().sort_values(ascending=False)
gen_pop.plot.bar(ax=axes[1], color=sns.color_palette('tab10', len(gen_pop)))
axes[1].set_title('Mean Popularity by Generation')
axes[1].set_ylabel('Mean Popularity Score')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'generation_analysis.png')
plt.show()

In [ ]:
# ============================================================
# 6C — Generation × Category crosstab (key sociolinguistic table)
# ============================================================
gen_cat = pd.crosstab(df['generation'], df['category'])

# Normalize to percentages for fair comparison
gen_cat_pct = gen_cat.div(gen_cat.sum(axis=1), axis=0) * 100

# Keep top 15 categories
top_cats = df['category'].value_counts().head(15).index
subset = gen_cat_pct[gen_cat_pct.columns.intersection(top_cats)]

fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(subset, annot=True, fmt='.1f', cmap='YlGnBu', ax=ax, linewidths=0.5)
ax.set_title('Generation × Category (% of generation total, top 15 categories)')
ax.set_ylabel('Generation')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'generation_category_heatmap.png')
plt.show()

print('\n--- Raw counts ---')
display(gen_cat[gen_cat.columns.intersection(top_cats)])

In [ ]:
# ============================================================
# 6D — Gen Z specific: which categories dominate?
# ============================================================
if 'Gen Z' in df['generation'].values:
    genz = df[df['generation'] == 'Gen Z']
    genz_cats = genz['category'].value_counts().head(15)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    genz_cats.plot.barh(ax=ax, color=sns.color_palette('mako', 15))
    ax.invert_yaxis()
    ax.set_xlabel('Count')
    ax.set_title('Top 15 Categories Used by Gen Z')
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'genz_categories.png')
    plt.show()
    
    # Drug terms: are they more generation-specific?
    drug_mask = df['category'].str.contains('drug', case=False, na=False)
    if drug_mask.any():
        drug_gen = df[drug_mask]['generation'].value_counts()
        print('\n🧪 Drug-related terms by generation:')
        display(drug_gen)
else:
    print('No Gen Z entries found.')

In [ ]:
# ============================================================
# 6E — Generation × Severity interaction
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))
gen_sev = df.groupby('generation')['severity'].agg(['mean','median','std','count'])
gen_sev_sorted = gen_sev.sort_values('mean', ascending=False)

ax.bar(range(len(gen_sev_sorted)), gen_sev_sorted['mean'],
       yerr=gen_sev_sorted['std'], capsize=5,
       color=sns.color_palette('tab10', len(gen_sev_sorted)))
ax.set_xticks(range(len(gen_sev_sorted)))
ax.set_xticklabels(gen_sev_sorted.index, rotation=45)
ax.set_ylabel('Mean Severity (± std)')
ax.set_title('Mean Severity by Generation')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'severity_by_generation.png')
plt.show()

display(gen_sev_sorted.round(2))

---
## 7. Content Flag Analysis

In [ ]:
# ============================================================
# 7A — Boolean flag summary
# ============================================================
flag_cols = ['safe_for_work', 'hate_speech', 'sexual', 'religious', 'family_related']
flag_summary = pd.DataFrame({
    'True_count': [df[c].sum() for c in flag_cols],
    'True_pct': [(df[c].sum()/len(df)*100) for c in flag_cols],
    'False_count': [(~df[c]).sum() for c in flag_cols],
}, index=flag_cols)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Counts
flag_summary['True_count'].plot.bar(ax=axes[0], color=['#2ecc71','#e74c3c','#e67e22','#9b59b6','#3498db'])
axes[0].set_title('Content Flag Counts (True)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(flag_summary['True_count']):
    axes[0].text(i, v + flag_summary['True_count'].max()*0.01, f'{v:,}', ha='center', fontsize=9)

# Percentages
flag_summary['True_pct'].plot.bar(ax=axes[1], color=['#2ecc71','#e74c3c','#e67e22','#9b59b6','#3498db'])
axes[1].set_title('Content Flag Percentages')
axes[1].set_ylabel('% of dataset')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'content_flags.png')
plt.show()

display(flag_summary.round(2))

In [ ]:
# ============================================================
# 7B — Flag co-occurrence matrix
# ============================================================
flag_matrix = df[flag_cols].astype(int)
co_occur = flag_matrix.T.dot(flag_matrix)

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(co_occur, dtype=bool), k=1)
sns.heatmap(co_occur, annot=True, fmt='d', cmap='Reds', ax=ax,
            linewidths=1, mask=None)
ax.set_title('Content Flag Co-occurrence Matrix')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'flag_cooccurrence.png')
plt.show()

In [ ]:
# ============================================================
# 7C — Hate speech analysis
# ============================================================
hs = df[df['hate_speech'] == True]
print(f'Hate speech entries: {len(hs):,} ({len(hs)/len(df)*100:.1f}%)\n')

if len(hs) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # By language
    hs['language'].value_counts().head(10).plot.barh(ax=axes[0], color='#e74c3c')
    axes[0].invert_yaxis()
    axes[0].set_title('Hate Speech by Language (top 10)')
    
    # By category
    hs['category'].value_counts().head(10).plot.barh(ax=axes[1], color='#c0392b')
    axes[1].invert_yaxis()
    axes[1].set_title('Hate Speech by Category (top 10)')
    
    # Severity of hate vs non-hate
    sns.violinplot(data=df, x='hate_speech', y='severity', ax=axes[2],
                   palette=['#2ecc71','#e74c3c'])
    axes[2].set_title('Severity: Hate Speech vs Not')
    axes[2].set_xticklabels(['Non-hate', 'Hate speech'])
    
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'hate_speech_analysis.png')
    plt.show()
    
    # Key finding: hate speech severity stats
    print('Hate speech severity stats:')
    display(df.groupby('hate_speech')['severity'].describe().round(2))

In [ ]:
# ============================================================
# 7D — Hate speech × Severity paradox analysis
# ============================================================
# "Interesting finding: Hate-speech categories may not always have the highest severity labels"
print('=== FINDING: Hate speech severity paradox ===')
print()
hs_mean_sev = df[df['hate_speech']==True]['severity'].mean()
non_hs_mean_sev = df[df['hate_speech']==False]['severity'].mean()
print(f'Mean severity of hate-speech entries:     {hs_mean_sev:.2f}')
print(f'Mean severity of non-hate-speech entries: {non_hs_mean_sev:.2f}')

# Mann-Whitney U test
u_stat, u_pval = stats.mannwhitneyu(
    df[df['hate_speech']==True]['severity'].dropna(),
    df[df['hate_speech']==False]['severity'].dropna(),
    alternative='two-sided'
)
print(f'\nMann-Whitney U test: U={u_stat:,.0f}, p={u_pval:.4e}')
print('Significant difference' if u_pval < 0.05 else 'No significant difference')

---
## 8. Etymology & Loanword Patterns

In [ ]:
# ============================================================
# 8A — Etymology keyword extraction
# ============================================================
# Extract source language/root indicators from etymology field
source_langs = []
etym_keywords = {
    'Sanskrit': r'\bsanskrit\b',
    'Arabic': r'\barabic\b',
    'Persian': r'\bpersian\b',
    'English': r'\benglish\b',
    'Latin': r'\blatin\b',
    'Turkic': r'\bturkic\b',
    'Austronesian': r'\baustronesian\b',
    'Hindi': r'\bhindi\b',
    'Urdu': r'\burdu\b',
    'Portuguese': r'\bportuguese\b',
    'French': r'\bfrench\b',
    'Greek': r'\bgreek\b',
    'Bantu': r'\bbantu\b',
    'Dravidian': r'\bdravidian\b',
    'Polynesian': r'\bpolynesian\b',
}

for kw, pattern in etym_keywords.items():
    mask = df['etymology'].str.contains(pattern, case=False, na=False)
    source_langs.append({'source': kw, 'count': mask.sum()})

etym_df = pd.DataFrame(source_langs).sort_values('count', ascending=False)
etym_df = etym_df[etym_df['count'] > 0]

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(etym_df['source'], etym_df['count'], color=sns.color_palette('Set3', len(etym_df)))
ax.invert_yaxis()
ax.set_xlabel('Count of entries mentioning this source')
ax.set_title('Etymological Source Languages/Families')
for i, v in enumerate(etym_df['count']):
    ax.text(v + etym_df['count'].max()*0.01, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'etymology_sources.png')
plt.show()

In [ ]:
# ============================================================
# 8B — English loanwords analysis
# ============================================================
eng_loan_mask = df['etymology'].str.contains(r'\benglish\b', case=False, na=False)
eng_loans = df[eng_loan_mask]

print(f'Entries with English etymological influence: {len(eng_loans):,} ({len(eng_loans)/len(df)*100:.1f}%)')

if len(eng_loans) > 5:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    eng_loans['language'].value_counts().head(10).plot.barh(ax=axes[0], color='#3498db')
    axes[0].invert_yaxis()
    axes[0].set_title('English Loanwords: Host Languages (top 10)')
    
    eng_loans['category'].value_counts().head(10).plot.barh(ax=axes[1], color='#2980b9')
    axes[1].invert_yaxis()
    axes[1].set_title('English Loanwords: Categories (top 10)')
    
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'english_loanwords.png')
    plt.show()

In [ ]:
# ============================================================
# 8C — Etymology text length (proxy for documentation depth)
# ============================================================
df['etymology_len'] = df['etymology'].str.len()

fig, ax = plt.subplots(figsize=(10, 5))
df['etymology_len'].hist(bins=50, ax=ax, color='#1abc9c', edgecolor='white')
ax.set_xlabel('Etymology Field Length (chars)')
ax.set_ylabel('Count')
ax.set_title('Etymology Documentation Depth')
ax.axvline(df['etymology_len'].median(), color='red', linestyle='--',
           label=f"Median={df['etymology_len'].median():.0f}")
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / 'etymology_length.png')
plt.show()

---
## 9. Word-Level Text Analysis

In [ ]:
# ============================================================
# 9A — Word length analysis
# ============================================================
df['word_len'] = df['word'].str.len()
df['transliteration_len'] = df['transliteration'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['word_len'].hist(bins=50, ax=axes[0], color='#e67e22', edgecolor='white')
axes[0].set_xlabel('Word Length (characters)')
axes[0].set_ylabel('Count')
axes[0].set_title('Word Length Distribution (native script)')
axes[0].axvline(df['word_len'].mean(), color='red', linestyle='--',
                label=f"Mean={df['word_len'].mean():.1f}")
axes[0].legend()

df['transliteration_len'].hist(bins=50, ax=axes[1], color='#d35400', edgecolor='white')
axes[1].set_xlabel('Transliteration Length (characters)')
axes[1].set_ylabel('Count')
axes[1].set_title('Transliteration Length Distribution')
axes[1].axvline(df['transliteration_len'].mean(), color='red', linestyle='--',
                label=f"Mean={df['transliteration_len'].mean():.1f}")
axes[1].legend()

plt.tight_layout()
plt.savefig(PLOT_DIR / 'word_length.png')
plt.show()

print(f"Longest word: '{df.loc[df['word_len'].idxmax(), 'word']}' ({df['word_len'].max()} chars, {df.loc[df['word_len'].idxmax(), 'language']})")
print(f"Shortest word: '{df.loc[df['word_len'].idxmin(), 'word']}' ({df['word_len'].min()} chars, {df.loc[df['word_len'].idxmin(), 'language']})")

In [ ]:
# ============================================================
# 9B — Mean word length by language
# ============================================================
wl_by_lang = df.groupby('language')['transliteration_len'].agg(['mean','count'])
wl_by_lang = wl_by_lang[wl_by_lang['count'] >= 10].sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(12, max(6, len(wl_by_lang.head(20))*0.35)))
wl_by_lang['mean'].head(20).plot.barh(ax=ax, color=sns.color_palette('copper', 20))
ax.invert_yaxis()
ax.set_xlabel('Mean Transliteration Length')
ax.set_title('Mean Word Length by Language (transliteration, min 10 entries)')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'word_length_by_language.png')
plt.show()

In [ ]:
# ============================================================
# 9C — Meaning / notes length analysis
# ============================================================
df['meaning_len'] = df['actual_meaning'].str.len()
df['notes_len'] = df['notes'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['meaning_len'].hist(bins=50, ax=axes[0], color='#8e44ad', edgecolor='white')
axes[0].set_xlabel('Meaning Length (chars)')
axes[0].set_title('Actual Meaning Length Distribution')

df['notes_len'].hist(bins=50, ax=axes[1], color='#2c3e50', edgecolor='white')
axes[1].set_xlabel('Notes Length (chars)')
axes[1].set_title('Notes Length Distribution')

plt.tight_layout()
plt.savefig(PLOT_DIR / 'text_field_lengths.png')
plt.show()

In [ ]:
# ============================================================
# 9D — Word cloud from transliterations
# ============================================================
try:
    from wordcloud import WordCloud
    
    text = ' '.join(df['transliteration'].dropna().astype(str))
    wc = WordCloud(width=1200, height=600, background_color='white',
                   max_words=200, colormap='inferno',
                   collocations=False).generate(text)
    
    fig, ax = plt.subplots(figsize=(15, 7))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('Word Cloud: Transliterations', fontsize=16)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'wordcloud_transliterations.png')
    plt.show()
except ImportError:
    print('wordcloud not installed. Run: pip install wordcloud')

In [ ]:
# ============================================================
# 9E — Word cloud from literal translations
# ============================================================
try:
    from wordcloud import WordCloud
    
    text_lit = ' '.join(df['literal_translation'].dropna().astype(str))
    wc2 = WordCloud(width=1200, height=600, background_color='black',
                    max_words=200, colormap='plasma',
                    collocations=False).generate(text_lit)
    
    fig, ax = plt.subplots(figsize=(15, 7))
    ax.imshow(wc2, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('Word Cloud: Literal Translations', fontsize=16, color='white')
    fig.patch.set_facecolor('black')
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'wordcloud_literal.png', facecolor='black')
    plt.show()
except ImportError:
    pass

In [ ]:
# ============================================================
# 9F — Most common literal translations (semantic themes)
# ============================================================
lit_words = df['literal_translation'].str.lower().str.strip()
lit_counts = lit_words.value_counts().head(30)

fig, ax = plt.subplots(figsize=(12, 8))
lit_counts.plot.barh(ax=ax, color=sns.color_palette('tab20', 30))
ax.invert_yaxis()
ax.set_xlabel('Count')
ax.set_title('Most Common Literal Translations (top 30)')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'common_literal_translations.png')
plt.show()

---
## 10. Cross-Linguistic Semantic Clustering (Embeddings)

In [ ]:
# ============================================================
# 10A — Embedding-based clustering
# ============================================================
# This section uses sentence-transformers for semantic clustering.
# On 100K entries, we sample for feasibility.

SAMPLE_SIZE = min(5000, len(df))  # Adjust based on your machine

try:
    from sentence_transformers import SentenceTransformer
    from sklearn.cluster import KMeans
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    
    print(f'Sampling {SAMPLE_SIZE:,} entries for embedding analysis...')
    sample = df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    
    # Build text for embedding
    sample['embed_text'] = (
        sample['actual_meaning'].fillna('') + ' ' +
        sample['notes'].fillna('') + ' ' +
        sample['literal_translation'].fillna('')
    ).str.strip()
    
    # Load model
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(sample['embed_text'].tolist(), show_progress_bar=True, batch_size=128)
    
    print(f'Embeddings shape: {embeddings.shape}')
    
    EMBEDDING_READY = True

except ImportError:
    print('sentence-transformers not installed. Run: pip install sentence-transformers')
    print('Skipping embedding analysis.')
    EMBEDDING_READY = False

In [ ]:
# ============================================================
# 10B — K-Means clustering
# ============================================================
if EMBEDDING_READY:
    N_CLUSTERS = 12  # Adjust as needed
    
    km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
    sample['cluster'] = km.fit_predict(embeddings)
    
    # Cluster sizes
    print('Cluster sizes:')
    display(sample['cluster'].value_counts().sort_index())
    
    # Top terms per cluster
    print('\n--- Top terms per cluster ---')
    for c in range(N_CLUSTERS):
        clust = sample[sample['cluster'] == c]
        top_cats = clust['category'].value_counts().head(3).index.tolist()
        top_words = clust['transliteration'].head(5).tolist()
        print(f"\nCluster {c} (n={len(clust)}): categories={top_cats}")
        print(f"  sample words: {top_words}")

In [ ]:
# ============================================================
# 10C — t-SNE visualization of clusters
# ============================================================
if EMBEDDING_READY:
    # Reduce to 2D with PCA first, then t-SNE (for speed)
    pca_50 = PCA(n_components=50, random_state=42)
    reduced_50 = pca_50.fit_transform(embeddings)
    
    tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
    coords = tsne.fit_transform(reduced_50)
    
    sample['tsne_x'] = coords[:, 0]
    sample['tsne_y'] = coords[:, 1]
    
    fig, ax = plt.subplots(figsize=(14, 10))
    scatter = ax.scatter(sample['tsne_x'], sample['tsne_y'],
                         c=sample['cluster'], cmap='tab20',
                         alpha=0.5, s=8)
    ax.set_title(f't-SNE of Semantic Embeddings ({N_CLUSTERS} clusters)')
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    plt.colorbar(scatter, ax=ax, label='Cluster')
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'tsne_clusters.png')
    plt.show()

In [ ]:
# ============================================================
# 10D — t-SNE colored by category (top 8)
# ============================================================
if EMBEDDING_READY:
    top8_cats = sample['category'].value_counts().head(8).index
    mask = sample['category'].isin(top8_cats)
    
    fig, ax = plt.subplots(figsize=(14, 10))
    for cat in top8_cats:
        sub = sample[sample['category'] == cat]
        ax.scatter(sub['tsne_x'], sub['tsne_y'], label=cat, alpha=0.5, s=12)
    
    ax.scatter(sample[~mask]['tsne_x'], sample[~mask]['tsne_y'],
               c='lightgray', alpha=0.15, s=5, label='other')
    ax.legend(fontsize=8, markerscale=3)
    ax.set_title('t-SNE Colored by Category (top 8)')
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'tsne_by_category.png')
    plt.show()

In [ ]:
# ============================================================
# 10E — Elbow method for optimal K
# ============================================================
if EMBEDDING_READY:
    inertias = []
    K_range = range(2, 21)
    for k in K_range:
        km_temp = KMeans(n_clusters=k, random_state=42, n_init=5)
        km_temp.fit(embeddings)
        inertias.append(km_temp.inertia_)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(K_range, inertias, 'bo-')
    ax.set_xlabel('Number of Clusters (K)')
    ax.set_ylabel('Inertia')
    ax.set_title('Elbow Method for Optimal K')
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'elbow_method.png')
    plt.show()

---
## 11. Network / Co-occurrence Analysis

In [ ]:
# ============================================================
# 11A — Category co-occurrence within languages
# ============================================================
# Which categories tend to co-occur within the same language?

from itertools import combinations

# Build co-occurrence
cat_pairs = Counter()
for lang, group in df.groupby('language'):
    cats_in_lang = group['category'].unique()
    for pair in combinations(sorted(cats_in_lang), 2):
        cat_pairs[pair] += 1

# Top pairs
top_pairs = cat_pairs.most_common(20)
print('Top 20 category pairs that co-occur across languages:')
for (a, b), count in top_pairs:
    print(f'  {a} ↔ {b}: {count} languages')

In [ ]:
# ============================================================
# 11B — Category co-occurrence heatmap
# ============================================================
top_cats_list = df['category'].value_counts().head(15).index.tolist()

cooccur_matrix = pd.DataFrame(0, index=top_cats_list, columns=top_cats_list)
for (a, b), count in cat_pairs.items():
    if a in top_cats_list and b in top_cats_list:
        cooccur_matrix.loc[a, b] = count
        cooccur_matrix.loc[b, a] = count

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cooccur_matrix, annot=True, fmt='d', cmap='Greens', ax=ax, linewidths=0.5)
ax.set_title('Category Co-occurrence Across Languages')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'category_cooccurrence.png')
plt.show()

In [ ]:
# ============================================================
# 11C — SFW vs NSFW by language
# ============================================================
sfw_by_lang = df.groupby('language')['safe_for_work'].mean() * 100
sfw_by_lang = sfw_by_lang.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 7))
top_n = min(25, len(sfw_by_lang))
colors = ['#2ecc71' if v > 50 else '#e74c3c' for v in sfw_by_lang.head(top_n)]
sfw_by_lang.head(top_n).plot.barh(ax=ax, color=colors)
ax.invert_yaxis()
ax.set_xlabel('% Safe for Work')
ax.set_title('SFW Percentage by Language (top 25)')
ax.axvline(50, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'sfw_by_language.png')
plt.show()

---
## 12. Statistical Tests & Correlations

In [ ]:
# ============================================================
# 12A — Correlation matrix for numeric & boolean fields
# ============================================================
numeric_cols = ['severity', 'popularity_score', 'toxicity_score',
                'word_len', 'transliteration_len', 'meaning_len', 'notes_len', 'etymology_len']
bool_num_cols = ['hate_speech', 'sexual', 'religious', 'family_related', 'safe_for_work']

corr_df = df[numeric_cols + bool_num_cols].copy()
for c in bool_num_cols:
    corr_df[c] = corr_df[c].astype(int)

corr = corr_df.corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'correlation_matrix.png')
plt.show()

In [ ]:
# ============================================================
# 12B — Chi-squared test: hate_speech vs category
# ============================================================
ct_hs_cat = pd.crosstab(df['hate_speech'], df['category'])
chi2, p_chi, dof, expected = stats.chi2_contingency(ct_hs_cat)
print(f'Chi-squared test: hate_speech × category')
print(f'  χ² = {chi2:,.1f}, df = {dof}, p = {p_chi:.2e}')
print(f'  → {"Significant" if p_chi < 0.05 else "Not significant"} association')

In [ ]:
# ============================================================
# 12C — Chi-squared test: generation × category
# ============================================================
ct_gen_cat = pd.crosstab(df['generation'], df['category'])
chi2_g, p_g, dof_g, _ = stats.chi2_contingency(ct_gen_cat)
print(f'Chi-squared test: generation × category')
print(f'  χ² = {chi2_g:,.1f}, df = {dof_g}, p = {p_g:.2e}')
print(f'  → {"Significant" if p_g < 0.05 else "Not significant"} association')

In [ ]:
# ============================================================
# 12D — Kruskal-Wallis: severity differs across tones?
# ============================================================
tone_groups = [g['severity'].dropna().values for _, g in df.groupby('tone') if len(g) > 5]
if len(tone_groups) >= 2:
    h_stat, h_p = stats.kruskal(*tone_groups)
    print(f'Kruskal-Wallis test: severity ~ tone')
    print(f'  H = {h_stat:.2f}, p = {h_p:.2e}')
    print(f'  → {"Significant" if h_p < 0.05 else "Not significant"} difference')

In [ ]:
# ============================================================
# 12E — Kruskal-Wallis: popularity differs across generations?
# ============================================================
gen_groups = [g['popularity_score'].dropna().values for _, g in df.groupby('generation') if len(g) > 5]
if len(gen_groups) >= 2:
    h_stat2, h_p2 = stats.kruskal(*gen_groups)
    print(f'Kruskal-Wallis test: popularity_score ~ generation')
    print(f'  H = {h_stat2:.2f}, p = {h_p2:.2e}')
    print(f'  → {"Significant" if h_p2 < 0.05 else "Not significant"} difference')

In [ ]:
# ============================================================
# 12F — Spearman correlations for ordinal data
# ============================================================
spearman_pairs = [
    ('severity', 'popularity_score'),
    ('severity', 'word_len'),
    ('popularity_score', 'transliteration_len'),
    ('toxicity_score', 'popularity_score'),
]

print('Spearman rank correlations:')
print(f'{"Pair":<45} {"rho":>6} {"p-value":>12}')
print('-' * 65)
for a, b in spearman_pairs:
    mask = df[[a, b]].dropna()
    if len(mask) > 10:
        rho, pval = stats.spearmanr(mask[a], mask[b])
        sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
        print(f'{a} × {b:<25} {rho:>6.3f} {pval:>12.2e} {sig}')

---
## 13. Paper-Ready Summary Tables & Figures

In [ ]:
# ============================================================
# 13A — Table 1: Dataset overview
# ============================================================
table1 = pd.DataFrame({
    'Metric': [
        'Total entries', 'Unique languages', 'Unique scripts',
        'Unique countries', 'Unique regions', 'Unique categories',
        'Unique target types', 'Unique tones', 'Unique generations',
        'Mean severity', 'Median severity', 'Mean popularity',
        'Hate speech entries (%)', 'Sexual content (%)',
        'Religious references (%)', 'Safe for work (%)',
    ],
    'Value': [
        f"{len(df):,}", df['language'].nunique(), df['script'].nunique(),
        df['country'].nunique(), df['region'].nunique(), df['category'].nunique(),
        df['target_type'].nunique(), df['tone'].nunique(), df['generation'].nunique(),
        f"{df['severity'].mean():.2f}", f"{df['severity'].median():.1f}",
        f"{df['popularity_score'].mean():.2f}",
        f"{df['hate_speech'].mean()*100:.1f}%",
        f"{df['sexual'].mean()*100:.1f}%",
        f"{df['religious'].mean()*100:.1f}%",
        f"{df['safe_for_work'].mean()*100:.1f}%",
    ]
})

print('TABLE 1: Dataset Overview')
display(table1)

In [ ]:
# ============================================================
# 13B — Table 2: Top 10 languages with summary stats
# ============================================================
table2 = df.groupby('language').agg(
    entries=('id', 'count'),
    categories=('category', 'nunique'),
    mean_severity=('severity', 'mean'),
    mean_popularity=('popularity_score', 'mean'),
    pct_hate_speech=('hate_speech', lambda x: x.mean()*100),
    pct_sexual=('sexual', lambda x: x.mean()*100),
    scripts=('script', 'nunique'),
).sort_values('entries', ascending=False).head(10).round(2)

print('TABLE 2: Top 10 Languages')
display(table2)

In [ ]:
# ============================================================
# 13C — Table 3: Category severity rankings
# ============================================================
table3 = df.groupby('category').agg(
    n=('id', 'count'),
    mean_severity=('severity', 'mean'),
    std_severity=('severity', 'std'),
    mean_popularity=('popularity_score', 'mean'),
    pct_hate=('hate_speech', lambda x: x.mean()*100),
    languages=('language', 'nunique'),
).sort_values('mean_severity', ascending=False)
table3 = table3[table3['n'] >= 5].head(20).round(2)

print('TABLE 3: Category Severity Rankings (n≥5, top 20)')
display(table3)

In [ ]:
# ============================================================
# 13D — Figure: Combined dashboard (4-panel)
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Panel 1: Severity distribution
df['severity'].value_counts().sort_index().plot.bar(
    ax=axes[0,0], color=sns.color_palette('Reds_d', df['severity'].nunique()))
axes[0,0].set_title('A) Severity Distribution')
axes[0,0].set_xlabel('Severity')
axes[0,0].set_ylabel('Count')

# Panel 2: Top 10 languages
df['language'].value_counts().head(10).plot.barh(
    ax=axes[0,1], color=sns.color_palette('viridis', 10))
axes[0,1].invert_yaxis()
axes[0,1].set_title('B) Top 10 Languages')
axes[0,1].set_xlabel('Entries')

# Panel 3: Generation × mean severity
gen_sev_plot = df.groupby('generation')['severity'].mean().sort_values()
gen_sev_plot.plot.barh(
    ax=axes[1,0], color=sns.color_palette('tab10', len(gen_sev_plot)))
axes[1,0].set_title('C) Mean Severity by Generation')
axes[1,0].set_xlabel('Mean Severity')

# Panel 4: Content flags
flag_pcts = pd.Series({
    'Hate speech': df['hate_speech'].mean()*100,
    'Sexual': df['sexual'].mean()*100,
    'Religious': df['religious'].mean()*100,
    'Family': df['family_related'].mean()*100,
    'SFW': df['safe_for_work'].mean()*100,
})
flag_pcts.plot.bar(
    ax=axes[1,1], color=['#e74c3c','#e67e22','#9b59b6','#3498db','#2ecc71'])
axes[1,1].set_title('D) Content Flag Percentages')
axes[1,1].set_ylabel('% of entries')
axes[1,1].tick_params(axis='x', rotation=45)

plt.suptitle('Cross-Linguistic Profanity Dataset: Overview Dashboard', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'dashboard_overview.png', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 13E — Figure: Sociolinguistic dashboard (4-panel)
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Panel 1: Popularity vs severity
jitter_s = df['severity'] + np.random.normal(0, 0.08, len(df))
jitter_p = df['popularity_score'] + np.random.normal(0, 0.08, len(df))
axes[0,0].scatter(jitter_s, jitter_p, alpha=0.1, s=5, c='#3498db')
z = np.polyfit(df['severity'].dropna(), df['popularity_score'].dropna(), 1)
p = np.poly1d(z)
x_l = np.linspace(df['severity'].min(), df['severity'].max(), 100)
axes[0,0].plot(x_l, p(x_l), 'r-', linewidth=2)
axes[0,0].set_title('A) Popularity vs Severity')
axes[0,0].set_xlabel('Severity')
axes[0,0].set_ylabel('Popularity')

# Panel 2: Popularity by generation (violin)
gen_order = df.groupby('generation')['popularity_score'].median().sort_values().index
sns.violinplot(data=df, x='generation', y='popularity_score', order=gen_order,
               ax=axes[0,1], palette='Set2', inner='quartile')
axes[0,1].set_title('B) Popularity by Generation')
axes[0,1].tick_params(axis='x', rotation=45)

# Panel 3: Toxicity by top 10 categories (boxplot)
top10_cats = df['category'].value_counts().head(10).index
sub10 = df[df['category'].isin(top10_cats)]
cat_order = sub10.groupby('category')['toxicity_score'].median().sort_values(ascending=False).index
sns.boxplot(data=sub10, y='category', x='toxicity_score', order=cat_order,
            ax=axes[1,0], palette='flare')
axes[1,0].set_title('C) Toxicity by Category (top 10)')

# Panel 4: Hate speech severity vs non-hate (split violin)
sns.violinplot(data=df, x='hate_speech', y='severity', ax=axes[1,1],
               palette=['#2ecc71','#e74c3c'], inner='quartile', split=False)
axes[1,1].set_title('D) Severity: Hate Speech vs Non-Hate')
axes[1,1].set_xticklabels(['Non-hate', 'Hate speech'])

plt.suptitle('Sociolinguistic Analysis Dashboard', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'dashboard_sociolinguistic.png', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 13F — Export summary stats to CSV
# ============================================================
# Language-level summary
lang_summary = df.groupby('language').agg(
    n=('id', 'count'),
    scripts=('script', lambda x: ', '.join(x.unique())),
    countries=('country', lambda x: ', '.join(x.unique())),
    categories=('category', 'nunique'),
    mean_severity=('severity', 'mean'),
    median_severity=('severity', 'median'),
    mean_popularity=('popularity_score', 'mean'),
    pct_hate=('hate_speech', lambda x: round(x.mean()*100, 1)),
    pct_sexual=('sexual', lambda x: round(x.mean()*100, 1)),
    pct_religious=('religious', lambda x: round(x.mean()*100, 1)),
    pct_sfw=('safe_for_work', lambda x: round(x.mean()*100, 1)),
    mean_toxicity=('toxicity_score', 'mean'),
    mean_word_len=('word_len', 'mean'),
).sort_values('n', ascending=False).round(2)

lang_summary.to_csv('language_summary_stats.csv')
print('Saved: language_summary_stats.csv')

# Category-level summary
cat_summary = df.groupby('category').agg(
    n=('id', 'count'),
    languages=('language', 'nunique'),
    mean_severity=('severity', 'mean'),
    mean_popularity=('popularity_score', 'mean'),
    pct_hate=('hate_speech', lambda x: round(x.mean()*100, 1)),
    pct_sexual=('sexual', lambda x: round(x.mean()*100, 1)),
    mean_toxicity=('toxicity_score', 'mean'),
    top_generation=('generation', lambda x: x.value_counts().index[0]),
).sort_values('n', ascending=False).round(2)

cat_summary.to_csv('category_summary_stats.csv')
print('Saved: category_summary_stats.csv')

display(lang_summary.head(10))

In [ ]:
# ============================================================
# 13G — Final report: all plot files
# ============================================================
plot_files = sorted(PLOT_DIR.glob('*.png'))
print(f'\n📊 Total plots generated: {len(plot_files)}')
for f in plot_files:
    print(f'  {f}')

print(f'\n📁 Summary CSVs: language_summary_stats.csv, category_summary_stats.csv')
print(f'\n✅ Analysis complete! {len(df):,} entries across {df["language"].nunique()} languages.')

---
## Appendix: Quick Reference

| Analysis | Section | Key Finding |
|----------|---------|-------------|
| Language coverage | §3 | Number of languages, scripts, geographic spread |
| Category taxonomy | §4 | Which insult types dominate cross-linguistically |
| Severity analysis | §5 | Mean severity by category/language; severity paradox |
| Toxicity scoring | §5G | Composite score = severity + 2×hate + sexual + religious |
| Generational slang | §6C-D | Gen Z categories; drug term specificity |
| Popularity dynamics | §6A-B | Popularity distributions by generation |
| Content flags | §7 | Hate speech, sexual, religious prevalence & co-occurrence |
| Hate speech paradox | §7D | Hate speech ≠ highest severity (statistical test) |
| Etymology patterns | §8 | English loanwords; etymological source families |
| Word morphology | §9 | Length distributions; longest/shortest words |
| Semantic clusters | §10 | Embedding-based groupings (liar/lazy/racial/drug) |
| Cross-cultural patterns | §11 | Category co-occurrence; SFW rates by language |
| Statistical tests | §12 | Chi-squared, Kruskal-Wallis, Spearman correlations |